# Bias–Variance Tradeoff

*Part of my classic ML interview preparation notes — written as a notebook so the
formulae and figures render.*

## Bias and Variance

When a model learns from data, it is trying to approximate the **true function** $f(x)$
that generates the data. But the model makes two kinds of error while doing this:

- **Bias**
  - How far off the model's prediction is from the true function.
  - Measures **systematic error** — i.e. how wrong the model is *on average*.
- **Variance**
  - How much the model's predictions fluctuate for different training sets.
  - Measures **sensitivity to noise**, or to the particular sample of data you trained on.

Therefore,

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \underbrace{\text{Irreducible Error}}_{\text{noise in the data — can't be fixed}}$$

| Concept | In words | Analogy |
|---------|----------|---------|
| **Total Error** | How wrong your model's predictions are, on average. | The total distance of your arrows from the bullseye. |
| **Bias$^2$** | How far the *average* prediction is from the truth — systematic error. | You always aim slightly off-center. |
| **Variance** | How much your predictions jump around (instability) if you retrain on different data. | Your arrows scatter all over the target. |
| **Noise** | Randomness in the data itself (can't be fixed). | Wind gusts you can't control. |

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# np.polyfit warns when high-degree fits are ill-conditioned; harmless for this demo.
warnings.filterwarnings("ignore", message="Polyfit may be poorly conditioned")
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (7, 4.5)

In [ ]:
# The dartboard analogy: bullseye = truth, each dot = one model's prediction.
scenarios = [
    ("Low Bias, Low Variance", (0.0, 0.0), 0.07),
    ("Low Bias, High Variance", (0.0, 0.0), 0.30),
    ("High Bias, Low Variance", (0.45, 0.35), 0.07),
    ("High Bias, High Variance", (0.45, 0.35), 0.30),
]

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for ax, (title, center, spread) in zip(axes.ravel(), scenarios):
    for r in (1.0, 0.66, 0.33):
        ax.add_patch(plt.Circle((0, 0), r, fill=False, color="gray", lw=1))
    ax.add_patch(plt.Circle((0, 0), 0.06, color="red"))  # bullseye = true value
    pts = rng.normal(loc=center, scale=spread, size=(40, 2))
    ax.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.7, color="steelblue")
    ax.set_title(title)
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect("equal")
    ax.axis("off")
fig.suptitle("Bias vs Variance — the dartboard analogy", fontsize=13)
plt.tight_layout()
plt.show()

### Building intuition for bias

When you train a model you only ever have **one** dataset. But in theory there could have
been infinitely many datasets, each sampled from the same true distribution.

Imagine our single dataset was generated by some true function $f(x)$ that we are trying to
approximate. We can first hold out a **test split** (say 20%). Then, from the remaining
training data, we resample many times — say 10 subsets — and fit a model on each. That gives
us 10 different predictions for every point in the test set. If we **average** those 10
predictions at a point and compare that average to the point's true value, the gap is the
**bias**.

If, for most test points, the *average* prediction sits far from the true value, the models
are systematically off — they are too simple to capture the underlying pattern. That is
**high bias** (underfitting), typical of, say, fitting a straight line to curved data.

> **Don't confuse this with cross-validation.** In CV we also train on different splits, but
> we evaluate each model on its *own* held-out fold and average a **performance metric**
> (accuracy, F1, …). For the bias picture above, we instead evaluate every model on the
> *same* test set and average the **predictions** to get an average prediction per point.

### The decomposition, formally

Suppose the data is generated as

$$y = f(x) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\ \sigma^2)$$

We train our model on different datasets $D_1, D_2, \ldots, D_n$, each giving a fitted model
$\hat{f}_{D}(x)$. The **average model prediction** over all datasets is

$$\bar{f}(x) = \mathbb{E}_D\!\left[\hat{f}_D(x)\right] = \frac{1}{n}\sum_{i=1}^{n} \hat{f}_{D_i}(x)$$

i.e. the expected prediction at $x$ if we retrained on infinitely many random datasets.
The expected squared error at a point $x$ then splits cleanly into three terms:

$$\mathbb{E}_D\!\left[(y - \hat{f}_D(x))^2\right] = \underbrace{(\bar{f}(x) - f(x))^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}_D\!\left[(\hat{f}_D(x) - \bar{f}(x))^2\right]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Noise}}$$

where

- $\mathbb{E}_D$ — expectation over different datasets $D$,
- $\hat{f}_D(x)$ — model trained on dataset $D$,
- $\bar{f}(x)$ — average prediction across datasets,
- $f(x)$ — the true function,
- $\sigma^2$ — variance of the noise $\varepsilon$ (irreducible).

### A closer look at variance

Bias asked: *where does the average prediction land?* Variance asks the complementary
question: *how much does an individual model jump around that average?* Using the same
thought experiment — many models trained on many resampled datasets — fix a test point $x$
and look at the spread of its predictions. If the 40 predictions cluster tightly, variance
is low; if they scatter, variance is high. Formally,

$$\text{Var}(x) = \mathbb{E}_D\!\left[(\hat{f}_D(x) - \bar{f}(x))^2\right]$$

This is exactly the everyday statistical variance, applied to predictions instead of raw
data: just as $\operatorname{Var}(X) = \mathbb{E}[(X - \mu)^2]$ measures how far samples sit
from their mean $\mu$, model variance measures how far each model's prediction sits from the
average prediction $\bar{f}(x)$. A model can have **low bias yet high variance** — capable of
hitting the truth *on average* but wildly unstable on any single dataset. That instability is
what we pay for with extra flexibility.

## The Bias–Variance Tradeoff

Bias and variance pull in opposite directions as you change model complexity:

| Model Complexity | Bias | Variance | Typical Example |
|------------------|------|----------|-----------------|
| Simple model (e.g. linear regression on nonlinear data) | High | Low | Underfitting |
| Complex model (e.g. deep neural net on a small dataset) | Low | High | Overfitting |

Total error is the **sum** of the two (plus irreducible noise), so it traces a U-shape: it
falls as we reduce bias, then rises again as variance takes over. The **sweet spot** is the
complexity that minimizes the total.

In [ ]:
# Schematic view: bias^2 falls, variance rises, total error is a U-curve.
complexity = np.linspace(1, 10, 200)
bias_sq = 3.0 / complexity        # decreases with complexity
variance = 0.05 * complexity      # increases with complexity
noise = np.full_like(complexity, 0.4)
total = bias_sq + variance + noise

plt.plot(complexity, bias_sq, label="Bias$^2$", lw=2)
plt.plot(complexity, variance, label="Variance", lw=2)
plt.plot(complexity, noise, "--", color="gray", label="Irreducible noise")
plt.plot(complexity, total, color="black", lw=2.5, label="Total error")
sweet = complexity[np.argmin(total)]
plt.axvline(sweet, color="green", ls=":", label=f"sweet spot (≈{sweet:.1f})")
plt.xlabel("Model complexity →")
plt.ylabel("Error")
plt.title("The Bias–Variance Tradeoff (schematic)")
plt.legend()
plt.tight_layout()
plt.show()

### Seeing it empirically

Let's actually *measure* the decomposition. We pick a true function
$f(x) = \sin(2\pi x)$ on a **fixed grid** of inputs, and — for each polynomial **degree**
(our complexity knob) — redraw the noisy labels many times and refit. From the resulting
cloud of predictions we estimate $\bar{f}(x)$, then compute $\text{Bias}^2$,
$\text{Variance}$, and the total. We fit with `numpy.polynomial.Polynomial.fit`, whose
scaled basis stays numerically stable even at high degree.

In [ ]:
from numpy.polynomial import Polynomial


def true_f(x):
    return np.sin(2 * np.pi * x)

sigma = 0.30          # std of the irreducible noise
n_train = 20          # fixed input grid size
n_datasets = 400      # how many times we redraw labels and refit
x_train = np.linspace(0, 1, n_train)       # same inputs every time...
x_test = np.linspace(0.02, 0.98, 100)      # ...evaluate on the interior
f_test = true_f(x_test)
degrees = range(1, 13)

bias2_by_deg, var_by_deg, err_by_deg = [], [], []
for d in degrees:
    preds = np.empty((n_datasets, x_test.size))
    for i in range(n_datasets):
        y_tr = true_f(x_train) + rng.normal(0, sigma, n_train)   # redraw noise
        preds[i] = Polynomial.fit(x_train, y_tr, d)(x_test)
    mean_pred = preds.mean(axis=0)                 # f-bar(x)
    bias2 = np.mean((mean_pred - f_test) ** 2)     # (f-bar - f)^2
    variance = np.mean(preds.var(axis=0))          # E_D[(f_hat - f-bar)^2]
    err = bias2 + variance + sigma ** 2            # expected test error
    bias2_by_deg.append(bias2)
    var_by_deg.append(variance)
    err_by_deg.append(err)

degs = np.array(list(degrees))
plt.plot(degs, bias2_by_deg, "o-", label="Bias$^2$")
plt.plot(degs, var_by_deg, "s-", label="Variance")
plt.axhline(sigma ** 2, ls="--", color="gray", label="Noise $\\sigma^2$")
plt.plot(degs, err_by_deg, "^-", color="black", label="Total error")
best = degs[int(np.argmin(err_by_deg))]
plt.axvline(best, color="green", ls=":", label=f"best degree = {best}")
plt.xlabel("Polynomial degree (model complexity)")
plt.ylabel("Error")
plt.title("Empirical bias\u2013variance decomposition")
plt.legend()
plt.tight_layout()
plt.show()

### Watching variance as "wiggle"

The curve above collapses everything into three numbers per degree. To *see* variance
directly, plot the fitted curves themselves. Below we keep the model class fixed and refit
**40 times**, each on the same inputs but with fresh noise. A low-degree model barely moves
(it's stubborn → high bias); a high-degree model thrashes around wildly from one noisy
sample to the next (it's fickle → high variance). The spread of the blue curves *is* the
variance.

In [ ]:
# Why 'variance' = wiggle: 40 models, each fit on the same inputs with fresh noise.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, d in zip(axes, (1, 12)):
    for _ in range(40):
        y_tr = true_f(x_train) + rng.normal(0, sigma, n_train)
        ax.plot(x_test, Polynomial.fit(x_train, y_tr, d)(x_test),
                color="steelblue", alpha=0.2)
    ax.plot(x_test, f_test, color="red", lw=2.5, label="true $f(x)$")
    label = "underfit (high bias)" if d == 1 else "overfit (high variance)"
    ax.set_title(f"degree {d}  \u2192  {label}")
    ax.set_ylim(-2, 2)
    ax.set_xlabel("x")
    ax.legend(loc="upper right")
axes[0].set_ylabel("y")
fig.suptitle("Same model class, 40 different noisy training sets", fontsize=12)
plt.tight_layout()
plt.show()

## Underfitting and Overfitting

Underfitting and overfitting are just the *practical faces* of bias and variance — what you
actually observe on training and validation curves.

**Underfitting** — the model is too simple to capture the pattern.

- High bias, low variance.
- High training error **and** high validation error.
- Example: linear regression on clearly nonlinear data.

**Overfitting** — the model is too complex for the amount of data, so it memorises noise.

- Low bias, high variance.
- Low training error but high validation error.
- Example: a high-degree polynomial threading through every training point.

| Behavior | Bias | Variance | Training error | Validation error |
|----------|------|----------|----------------|------------------|
| Underfitting | High | Low | High | High |
| Good fit | Balanced | Balanced | Low | Low |
| Overfitting | Low | High | Low | High |

## Detecting bias and variance in practice

In theory bias and variance are defined over infinitely many datasets and a *known* true
function. In practice we have neither — so we **infer** them from the gap between training and
validation performance:

- **High bias (underfitting):** high training loss, high validation loss, and a *small* gap
  between them. The model can't even fit the data it has seen.
- **High variance (overfitting):** low training loss but high validation loss — a *large*
  train–validation gap. The model fits the training set far better than anything new.
- **Balanced:** low training loss, low validation loss, small gap — good generalization.

The single most useful diagnostic is to plot **training and validation error against model
complexity**: training error falls monotonically, while validation error first falls and then
rises. The minimum of the validation curve marks the best bias–variance tradeoff.

In [ ]:
# Training error falls monotonically; validation error is U-shaped. Reuses the setup above.
train_err, val_err = [], []
for d in degrees:
    tr, va = [], []
    for _ in range(n_datasets):
        y_tr = true_f(x_train) + rng.normal(0, sigma, n_train)
        model = Polynomial.fit(x_train, y_tr, d)
        tr.append(np.mean((model(x_train) - y_tr) ** 2))    # error on its OWN training data
        va.append(np.mean((model(x_test) - f_test) ** 2))   # error vs the truth on unseen x
    train_err.append(np.mean(tr))
    val_err.append(np.mean(va))

degs = np.array(list(degrees))
best = degs[int(np.argmin(val_err))]
plt.plot(degs, train_err, "o-", label="training error")
plt.plot(degs, val_err, "s-", label="validation error")
plt.axvline(best, color="green", ls=":", label=f"best degree = {best}")
plt.axvspan(degs[0], best, alpha=0.06, color="orange")
plt.axvspan(best, degs[-1], alpha=0.06, color="purple")
ymax = max(val_err)
plt.text(degs[0] + 0.2, ymax * 0.85, "underfitting\n(high bias)", fontsize=9)
plt.text(degs[-1] - 0.2, ymax * 0.85, "overfitting\n(high variance)", fontsize=9, ha="right")
plt.xlabel("Polynomial degree (model complexity)")
plt.ylabel("Mean squared error")
plt.title("Training vs validation error")
plt.legend()
plt.tight_layout()
plt.show()

## Total error in practice

The theoretical quantity $\mathbb{E}\!\left[(y - \hat{f}(x))^2\right]$ can't be measured
directly — we don't know the true distribution. In practice we approximate it with a **loss
function**, evaluated on data we *do* have:

| Model | Typical loss function |
|-------|-----------------------|
| Linear regression | Mean Squared Error (MSE) |
| Logistic regression | Cross-Entropy (CE) |
| Neural networks | MSE or Cross-Entropy |
| LLMs | Token-level Cross-Entropy |

So the loss function is our practical stand-in for total error. **Training loss** tells us how
well the model fits data it has already seen; **validation loss** estimates how well it
generalizes to data it hasn't. The gap between them is exactly the bias-vs-variance signal
from the previous section.

## Cross-Validation

We want to estimate how well a model generalizes to **unseen** data. A single train–test
split can mislead us: the score depends heavily on *which* points happened to land in the
test set. **Cross-validation** gives a more robust estimate by rotating the held-out set.

**k-fold cross-validation.** Split the data into $k$ folds. For each fold $i$:

1. Hold out fold $i$ as validation data.
2. Train on the other $k-1$ folds.
3. Score the model on the held-out fold.

Then average the scores:

$$\text{CV score} = \frac{1}{k}\sum_{i=1}^{k} \text{Metric}_i$$

Because every point serves as validation data exactly once, the estimate depends far less on
any single lucky or unlucky split.

In [ ]:
# A 5-fold split: each row trains on the blue blocks and validates on the orange one.
k, n = 5, 20
fig, ax = plt.subplots(figsize=(8, 3.2))
fold_size = n / k
for i in range(k):                       # i = which fold is held out for validation
    for j in range(k):                   # j = each block along the row
        is_val = (j == i)
        ax.barh(i, fold_size, left=j * fold_size, height=0.7,
                color="orange" if is_val else "steelblue",
                edgecolor="white")
    ax.text(n + 0.4, i, f"fold {i + 1}", va="center", fontsize=9)
ax.set_yticks([])
ax.set_xlim(0, n + 3)
ax.set_xlabel("data points")
ax.invert_yaxis()
ax.set_title("5-fold cross-validation  (orange = validation, blue = training)")
plt.tight_layout()
plt.show()

### Why cross-validation differs from the theoretical bias–variance setup

It's worth being precise about how these two "train many models" procedures differ, because
they look similar but answer different questions.

- In the **theoretical decomposition**, we evaluate every model on the *same fixed test set*.
  Holding the test set constant is what lets us isolate how predictions move *purely* because
  the training data changed — that's the whole point of separating bias from variance.
- In **cross-validation**, the validation set *rotates*. We're no longer isolating bias and
  variance; we're trying to get a trustworthy estimate of real-world performance from one
  finite dataset, without betting everything on a single arbitrary split.

In short: the theoretical setup sacrifices realism to *isolate* bias and variance;
cross-validation sacrifices that purity to get a *robust* estimate of generalization error.

### Refitting on the full dataset

Once cross-validation shows stable, acceptable performance, we usually **don't** ship one of
the fold models. Each of those only ever saw $k-1$ folds of the data. Instead, with the
architecture and hyperparameters now chosen, we train one final model **from scratch on the
entire dataset**.

Why? Model selection is already done, so there's no reason to keep holding data back. Training
on everything lets the final model see more examples, capture more signal, and reduce sampling
noise — a stronger model than any single fold produced. This step goes by *final refit*,
*refit on full data*, or *production training*.

> **Key idea:** cross-validation helps us *choose* the model; the final refit helps us *build*
> the best version of the model we chose.

## How to move the needle

| To reduce **bias** (fix underfitting) | To reduce **variance** (fix overfitting) |
|---------------------------------------|------------------------------------------|
| Use a more expressive model / add features | Use a simpler model / drop features |
| Add polynomial or interaction terms | Gather more training data |
| Train longer; reduce regularization | Add regularization (L1/L2, dropout) |
| Reduce restrictive assumptions | Ensembling / bagging, early stopping |

## Key takeaways

- **Bias** = error from wrong assumptions (too simple) → underfitting; **variance** = error
  from sensitivity to the training sample (too complex) → overfitting.
- They **trade off**: lowering one usually raises the other, so total error is a U-curve and
  the goal is the complexity that minimizes the *sum*.
- **Irreducible noise** $\sigma^2$ sets a floor — no model can beat it.
- **More data** mainly shrinks *variance*; a **better model class / features** mainly shrinks
  *bias*.